# UCS420: Cognitive Computing — Assignment 4

## A Cognitive FAQ System Using Pandas (Nova 2.0)

Name: Vishu Verma  
Roll number: 1024170242

## Question 1 — Build the personalised knowledge base

Four fixed entries, plus two built from the **last two digits** of the roll number `1024170242`.

The last two digits are `4` and `2`:

- digit 4 -> `["billing", "account", "general"][4 % 3]` = `["billing", "account", "general"][1]` = **account**
- digit 2 -> `["billing", "account", "general"][2 % 3]` = `["billing", "account", "general"][2]` = **general**

So one entry is written for the *account* category and one for the *general* category.

In [1]:
import pandas as pd

roll_number = "1024170242"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# work out which category each of the last two digits maps to
categories = ["billing", "account", "general"]
last_two = [int(d) for d in roll_number[-2:]]

for digit in last_two:
    print("digit", digit, "-> category[", digit, "% 3 =", digit % 3, "] =", categories[digit % 3])

digit 4 -> category[ 4 % 3 = 1 ] = account
digit 2 -> category[ 2 % 3 = 2 ] = general


In [2]:
# my two entries, written to match the categories worked out above
my_entries = [
    {"question": "how do i update my registered mobile number",
     "answer": "Open Profile > Contact Details, type the new number and confirm the OTP sent to it.",
     "keywords": "mobile number update", "category": categories[last_two[0] % 3]},   # account
    {"question": "where is your office located",
     "answer": "Our office is on Thapar University Road, Patiala, Punjab.",
     "keywords": "office location address", "category": categories[last_two[1] % 3]},  # general
]

df = pd.DataFrame(fixed_entries + my_entries)
df

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,"Open Profile > Contact Details, type the new n...",mobile number update,account
5,where is your office located,"Our office is on Thapar University Road, Patia...",office location address,general


## Question 2 — Generate and score a hypothesis

The scoring function takes a query, scores every entry, and returns all the entries that matched,
ranked by confidence.

Confidence = (how many words of the query the entry recognised) / (how many words the query has),
so it is always between 0 and 1.

In [3]:
# words like "how" and "my" sit in almost every question, so they would make
# every entry look like a match -- they are dropped before scoring
STOPWORDS = {"a", "an", "the", "is", "are", "do", "does", "how", "what", "when", "where",
             "which", "i", "my", "me", "to", "of", "for", "can", "you", "your", "and",
             "in", "on", "it"}


def clean(text):
    words = {w.strip("?.,!") for w in text.lower().split()}
    return (words - STOPWORDS) or words   # a query made only of filler is kept as it is


def score_entry(query, row):
    """Confidence that one FAQ row answers the query."""
    query_words = clean(query)
    known_words = clean(row["keywords"]) | clean(row["question"])
    matched = query_words & known_words
    return len(matched) / len(query_words)


def search(query, df):
    """All entries that match the query, best confidence first."""
    scored = df.copy()
    scored["confidence"] = scored.apply(lambda row: score_entry(query, row), axis=1)
    matches = scored[scored["confidence"] > 0]
    return matches.sort_values("confidence", ascending=False)[["question", "answer", "category", "confidence"]]

In [4]:
query = "how much does the fee cost"
print("Query:", query)
search(query, df)

Query: how much does the fee cost


,question,answer,category,confidence
0,what is the annual fee,The annual fee is Rs 500.,billing,0.666667
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.333333


In [5]:
query = "i want to change my mobile number"
print("Query:", query)
search(query, df)

Query: i want to change my mobile number


,question,answer,category,confidence
4,how do i update my registered mobile number,"Open Profile > Contact Details, type the new n...",account,0.5


## Question 3 — `same_category(category_name, df)`

In [6]:
def same_category(category_name, df):
    """All questions that belong to a given category."""
    return df[df["category"] == category_name]["question"]


# called with the category of one of my personalised entries (digit 4 -> account)
my_category = categories[last_two[0] % 3]
print("Questions in the", my_category, "category:")
print(same_category(my_category, df))

Questions in the account category:
1                          how to reset password
4    how do i update my registered mobile number
Name: question, dtype: str


## Question 4 — Add a keyword and save to CSV

One entry is picked, the user types a new keyword, it is added to that entry's keywords,
and the whole DataFrame is written to `1024170242_faq_data.csv`.

In [7]:
# the entry being updated -- my "account" entry about the mobile number
target = 4
print("Chosen entry:", df.loc[target, "question"])
print("Keywords before:", df.loc[target, "keywords"])

new_keyword = input("Enter a new keyword to add: ")
df.loc[target, "keywords"] = df.loc[target, "keywords"] + " " + new_keyword.strip()

print("Keywords after :", df.loc[target, "keywords"])

Enter a new keyword to add: otp
Chosen entry: how do i update my registered mobile number
Keywords before: mobile number update


Keywords after : mobile number update otp


In [8]:
file_name = roll_number + "_faq_data.csv"
df.to_csv(file_name, index=False)
print("Saved to", file_name)

pd.read_csv(file_name)

Saved to 1024170242_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,"Open Profile > Contact Details, type the new n...",mobile number update otp,account
5,where is your office located,"Our office is on Thapar University Road, Patia...",office location address,general


## Question 5 — Entries per category using `groupby`

In [9]:
print("Number of FAQ entries per category:")
print(df.groupby("category").size())

Number of FAQ entries per category:
category
account    2
billing    2
general    2
dtype: int64


## Question 6 — Scoring that does not hide a tie

Same scoring as Q2, but now the top score is looked at first. If more than one entry shares it,
every one of them is printed instead of quietly returning the first.

In [10]:
def search_best(query, df):
    """Print every entry that ties for the best confidence, instead of picking one."""
    scored = df.copy()
    scored["confidence"] = scored.apply(lambda row: score_entry(query, row), axis=1)
    matches = scored[scored["confidence"] > 0]

    print("Query:", query)
    if matches.empty:
        print("No entry matched this query.")
        return matches

    best = matches["confidence"].max()
    top = matches[matches["confidence"] == best]

    if len(top) > 1:
        print("TIE --", len(top), "entries share the best confidence of", round(best, 3), ":")
    else:
        print("One clear best match, confidence", round(best, 3), ":")

    for _, row in top.iterrows():
        print("  Q:", row["question"])
        print("  A:", row["answer"], "\n")

    return top[["question", "answer", "category", "confidence"]]

In [11]:
# a query that ties -- "fee" is a keyword of both billing entries, so both score 1.0
search_best("fee", df)

Query: fee
TIE -- 2 entries share the best confidence of 1.0 :
  Q: what is the annual fee
  A: The annual fee is Rs 500. 

  Q: how can i pay the fee
  A: You can pay via UPI, card, or net banking. 



,question,answer,category,confidence
0,what is the annual fee,The annual fee is Rs 500.,billing,1.0
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1.0


In [12]:
# a query that does not tie -- only the password entry recognises these words
search_best("how to reset password", df)

Query: how to reset password
One clear best match, confidence 1.0 :
  Q: how to reset password
  A: Go to Settings > Reset Password. 



,question,answer,category,confidence
1,how to reset password,Go to Settings > Reset Password.,account,1.0
